# 개별종목 조합D — LogisticRegression

`기본모델/01.LogisticRegression.ipynb`과 같은 `models.logistic.build_logistic_baseline`을 가져오고
조합D 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.logistic import build_logistic_baseline  # noqa: E402

MODEL_NAME = 'LogisticRegression'
MODEL_BUILDER = build_logistic_baseline


In [2]:
# 2. 조합D의 피처 값만 지정합니다.
import json

COMBINATION = 'D'
FEATURE_COLUMNS = (
    'atr_ratio',
    'hv_20',
    'range_1',
    'range_20',
    'bb_bandwidth',
    'volume_z_20',
    'turnover_20',
    'log_amihud_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합D 피처: ('atr_ratio', 'hv_20', 'range_1', 'range_20', 'bb_bandwidth', 'volume_z_20', 'turnover_20', 'log_amihud_20')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.5099,0.5012,0.0087,0.3158,0.3674,0.1008,0.4003,0.0937,0.1899
1,2,balanced,980,20150123,20150421,0.4061,0.3978,0.0083,0.3540,0.3716,0.0706,0.3898,0.1664,0.2656
2,3,balanced,1210,20151228,20160328,0.3799,0.3762,0.0038,0.3644,0.3701,0.0591,0.3859,0.2296,0.3083
3,4,balanced,1439,20161202,20170228,0.4659,0.4617,0.0042,0.3368,0.3729,0.0909,0.4086,0.1166,0.2191
4,5,balanced,1669,20171113,20180207,0.4154,0.3901,0.0254,0.3839,0.3935,0.1000,0.3940,0.3199,0.3686
5,6,balanced,1899,20181024,20190118,0.3898,0.3725,0.0173,0.3906,0.3925,0.0939,0.4068,0.4235,0.4007
6,7,balanced,2129,20190930,20191224,0.4556,0.4781,-0.0225,0.3115,0.3511,0.0527,0.4050,0.1301,0.2291
7,8,balanced,2359,20200902,20201130,0.3782,0.3476,0.0306,0.3632,0.3648,0.0540,0.3902,0.2240,0.3042
8,9,balanced,2589,20210806,20211105,0.4000,0.3914,0.0086,0.3830,0.4125,0.1083,0.3918,0.2014,0.2977
9,10,balanced,2818,20220714,20221012,0.3320,0.3454,-0.0135,0.3145,0.3480,0.0207,0.3705,0.1045,0.1903


,OOS 폴드 평균
accuracy,0.4095
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0126
macro_f1,0.3556
balanced_accuracy,0.3765
mcc,0.0770
pr_auc_macro_ovr,0.3957
down_recall,0.2126
core_harmonic_mean,0.2869


재실행 명령: python scripts/run_stock_model_experiment.py
